# Lesson 06 — The Full Pipeline

In Lessons 03–05 we ran each step manually. Now we connect them with **job dependencies** — Batch ensures Job 2 only starts after Job 1 succeeds.

```
Upload images → [Job 1: generate captions] → [Job 2: embed captions] → Embeddings in S3 Vectors
                        ↑ must SUCCEED before Job 2 even starts
```

This is the foundation of every ML batch pipeline.

## How Batch job dependencies work

```python
job1 = batch.submit_job(jobName="caption", ...)          # submitted now

job2 = batch.submit_job(
    jobName  = "embed",
    dependsOn= [{"jobId": job1["jobId"], "type": "N_TO_N"}],  # waits for job1
    ...
)
```

Both jobs are submitted instantly. Job 2 sits in `PENDING` state until Job 1 reaches `SUCCEEDED`. If Job 1 fails, Job 2 is automatically cancelled — **you never get stale embeddings from failed captions**.

## Step 1 — Load environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")
print(f"Bucket: {os.environ['S3_BUCKET']}")
print(f"Queue : {os.environ['BATCH_JOB_QUEUE']}")

## Step 2 — Run the full pipeline

`run_pipeline.py` uploads the images, submits both jobs with dependencies, and polls until done.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "run_pipeline.py", "--images-dir", "assets/images", "--batch-stem", "sample"],
)
print("Exit code:", result.returncode)

## Step 3 — Verify: check that both outputs exist

In [ ]:
import boto3, os

s3     = boto3.client("s3")
s3vectors = boto3.client("s3vectors")
bucket = os.environ["S3_BUCKET"]

# Check 1 — captions manifest exists in ordinary S3
try:
    s3.head_object(Bucket=bucket, Key="captions/sample/manifest.json")
    captions_found = True
except Exception:
    captions_found = False
print(f"{'Caption manifest':20s}: {'✅' if captions_found else '❌ not found'}")

# Check 2 — S3 Vectors returns matches for this image batch
resp = s3vectors.query_vectors(
    vectorBucketName=os.environ["S3_VECTOR_BUCKET"],
    indexName=os.environ["S3_VECTOR_INDEX"],
    topK=1,
    queryVector={"float32": [1.0] + [0.0] * 511},
    returnMetadata=True,
)
vectors_found = len(resp.get("vectors", [])) > 0
print(f"{'S3 Vectors entries':20s}: {'✅' if vectors_found else '❌ not found'}")

## Key Takeaway

> **`dependsOn` = declarative ordering.** You describe the dependencies; Batch runs the graph.
> For production pipelines (10+ steps, branching, retries), tools like Apache Airflow or AWS Step Functions build on this same concept but add monitoring and retry logic.

---

## Next lesson → [07 — Scale & Cost](../07-scale-and-cost/notebook.ipynb)

We'll process multiple image batches in parallel using Batch **array jobs**, and calculate the real cost.